# 06 클러스터링 방법

**6종 임베딩** × **4종 클러스터링** = **24조합** 품질 비교 (실루엣 ↑, Davies-Bouldin ↓)

| 임베딩 | 클러스터링 |
|--------|-----------|
| PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST | KMeans, HAC, GMM, DBSCAN |

최적 조합 라벨 → `ml_cluster_type_family.parquet` 저장


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.clustering_experiments import run_embedding_clustering_grid, select_best_combo, add_joint_rank

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')
pivot = dfw.pivot_table(index=['type', 'family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = pivot.values.astype(float)
K = 4
N_COMPONENTS = 10
print('series:', X_raw.shape[0], '| weeks:', X_raw.shape[1], '| K:', K)



series: 165 | weeks: 242 | K: 4


In [2]:
# 6 임베딩 × 4 클러스터링 = 24조합 (FastDTW·AE·GAF-CNN·TS2Vec·PatchTST 포함)
quality_df, label_cache = run_embedding_clustering_grid(
    X_raw, k=K, n_components=N_COMPONENTS,
)
quality_ranked = add_joint_rank(quality_df, k=K)
print('=== K=4 조합: 실루엣·DB Index 동시 순위 (rank_score = sil_rank + db_rank, 낮을수록 우수) ===')
display(quality_ranked.round(4))



[embedding] PCA


  PCA+KMeans: silhouette=0.49409916553149, db=0.6896313697646943, n_clusters=4
  PCA+HAC: silhouette=0.8564530877738853, db=0.7436324163163133, n_clusters=4
  PCA+GMM: silhouette=0.2473123147734971, db=1.3003375769669379, n_clusters=4
  PCA+DBSCAN: silhouette=nan, db=nan, n_clusters=1
[embedding] FastDTW


  FastDTW+KMeans: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+HAC: silhouette=0.8173942337871033, db=0.5003616061201555, n_clusters=4
  FastDTW+GMM: silhouette=0.8123490350068091, db=0.5188366934371917, n_clusters=4
  FastDTW+DBSCAN: silhouette=0.7339057572674218, db=0.27805358777026506, n_clusters=2
[embedding] AE


  AE+KMeans: silhouette=0.8429715037345886, db=0.6183879793141012, n_clusters=4
  AE+HAC: silhouette=0.8097119927406311, db=0.5429997379134139, n_clusters=4
  AE+GMM: silhouette=0.7646256685256958, db=0.5948829989051201, n_clusters=4
  AE+DBSCAN: silhouette=0.7548282742500305, db=0.22601625554446853, n_clusters=2
[embedding] GAF-CNN


  GAF-CNN+KMeans: silhouette=0.38246774673461914, db=1.1189489076092356, n_clusters=4
  GAF-CNN+HAC: silhouette=0.3628520965576172, db=1.3206303913776807, n_clusters=4
  GAF-CNN+GMM: silhouette=0.21614660322666168, db=1.7814021936273008, n_clusters=4
  GAF-CNN+DBSCAN: silhouette=0.39146390557289124, db=0.9632872907432207, n_clusters=2
[embedding] TS2Vec


  TS2Vec+KMeans: silhouette=0.8304817080497742, db=0.4866095649879274, n_clusters=4
  TS2Vec+HAC: silhouette=0.7813594937324524, db=0.5615847328974254, n_clusters=4
  TS2Vec+GMM: silhouette=0.6580440998077393, db=0.6943064785323935, n_clusters=4
  TS2Vec+DBSCAN: silhouette=0.8573176860809326, db=0.2097741245609104, n_clusters=2
[embedding] PatchTST


  PatchTST+KMeans: silhouette=0.531990647315979, db=0.7065659588832756, n_clusters=4
  PatchTST+HAC: silhouette=0.48002365231513977, db=0.7192579420484106, n_clusters=4
  PatchTST+GMM: silhouette=0.3555922210216522, db=1.1268261765160088, n_clusters=4
  PatchTST+DBSCAN: silhouette=0.7110080718994141, db=0.5814078413453507, n_clusters=6
=== K=4 조합: 실루엣·DB Index 동시 순위 (rank_score = sil_rank + db_rank, 낮을수록 우수) ===


,method,embedding,clustering,n_clusters,silhouette,davies_bouldin,sil_rank,db_rank,rank_score
0,TS2Vec+KMeans,TS2Vec,KMeans,4,0.8305,0.4866,3.0,1.0,4.0
1,FastDTW+HAC,FastDTW,HAC,4,0.8174,0.5004,4.0,2.0,6.0
2,FastDTW+KMeans,FastDTW,KMeans,4,0.8123,0.5188,5.5,3.5,9.0
3,FastDTW+GMM,FastDTW,GMM,4,0.8123,0.5188,5.5,3.5,9.0
4,AE+KMeans,AE,KMeans,4,0.8430,0.6184,2.0,8.0,10.0
5,AE+HAC,AE,HAC,4,0.8097,0.5430,7.0,5.0,12.0
6,TS2Vec+HAC,TS2Vec,HAC,4,0.7814,0.5616,8.0,6.0,14.0
7,PCA+HAC,PCA,HAC,4,0.8565,0.7436,1.0,13.0,14.0
8,AE+GMM,AE,GMM,4,0.7646,0.5949,9.0,7.0,16.0
9,TS2Vec+GMM,TS2Vec,GMM,4,0.6580,0.6943,10.0,10.0,20.0


In [3]:
best = select_best_combo(quality_df, k=K)
print('최적 조합 (실루엣+DB 동시 순위, K=4):', best['method'])
print(f"rank_score={float(best['rank_score']):.1f}, n_clusters={int(best['n_clusters'])}, silhouette={float(best['silhouette']):.4f}, davies_bouldin={float(best['davies_bouldin']):.4f}")

pca_km = quality_df[quality_df['method'] == 'PCA+KMeans'].iloc[0]
print('비교 PCA+KMeans:', f"sil={float(pca_km['silhouette']):.4f}, db={float(pca_km['davies_bouldin']):.4f}, n_clusters={int(pca_km['n_clusters'])}")

labels_best = label_cache[(best['embedding'], best['clustering'])]
out = meta.copy()
out['ML_CLUSTER'] = labels_best + 1
out['embedding_method'] = best['embedding']
out['clustering_method'] = best['clustering']
out.to_parquet(ML_CLUSTER, index=False)
quality_df.to_csv(DATA_PROCESSED / 'clustering_quality.csv', index=False)
quality_ranked.to_csv(DATA_PROCESSED / 'clustering_quality_ranked.csv', index=False)
print('저장:', ML_CLUSTER)
out.head()



최적 조합 (실루엣+DB 동시 순위, K=4):

 TS2Vec+KMeans
rank_score=4.0, n_clusters=4, silhouette=0.8305, davies_bouldin=0.4866
비교 PCA+KMeans: sil=0.4941, db=0.6896, n_clusters=4
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\ml_cluster_type_family.parquet


,type,family,ML_CLUSTER,embedding_method,clustering_method
0,A,AUTOMOTIVE,1,TS2Vec,KMeans
1,A,BABY CARE,1,TS2Vec,KMeans
2,A,BEAUTY,1,TS2Vec,KMeans
3,A,BEVERAGES,2,TS2Vec,KMeans
4,A,BOOKS,1,TS2Vec,KMeans


## 분석 요약

### 선정 기준
- **실루엣 ↑** + **Davies-Bouldin ↓** 동시 반영 (`rank_score = sil_rank + db_rank`, **K=4 조합만**)
- DBSCAN(K≠4)은 제외 — SBC 4분류와 맞추기 위함

### K=4 조합 비교 (동시 순위)
| rank_score | 조합 | Silhouette | DB Index |
|------------|------|-----------|----------|
| **4** | **TS2Vec+KMeans** | 0.831 | **0.487** (DB 1위) |
| 6 | FastDTW+HAC | 0.817 | 0.500 |
| 14 | PCA+HAC | **0.856** (실루엣 1위) | **0.744** (DB 나쁨) |
| **21** | **PCA+KMeans** | **0.494** | **0.690** |

→ **PCA+KMeans는 K=4 조합 중 하위권.** 실루엣만 보면 PCA+HAC가 1위지만 DB Index가 0.744로 크게 손해 → 동시 순위에서 밀림.

### 저장
- 최적: **TS2Vec+KMeans** → `ml_cluster_type_family.parquet`